# Task 2 — Data Anonymisation
This notebook demonstrates the transformation workflow. Keep the original customer workbook private; do not commit raw customer data or customer-level exports to a public repository.

The resulting UUID-based data is de-identified, not guaranteed to be anonymous.

In [ ]:
import pandas as pd
import uuid

df = pd.read_excel('mobile_customers.xlsx')  # local/private input; intentionally not tracked by Git
df.shape, df.columns.tolist()

In [ ]:
# Remove direct identifiers and sensitive payment fields
sensitive_columns = ['username', 'name', 'address', 'email', 'birthdate', 'credit_card_number', 'security_code', 'expiry']
anon_df = df.drop(columns=[c for c in sensitive_columns if c in df.columns])

# Generalise age and salary
if 'age' in anon_df:
    anon_df['age'] = pd.cut(anon_df['age'], bins=[0,25,35,45,55,float('inf')], labels=['Up to 25','26–35','36–45','46–55','56+'])
if 'salary' in anon_df:
    anon_df['salary'] = pd.cut(anon_df['salary'], bins=[-1,50000,100000,150000,200000,250000], labels=['0–50K','50K–100K','100K–150K','150K–200K','200K–250K'])

In [ ]:
# Replace customer IDs and remove additional quasi-identifiers
if 'customer_id' in anon_df:
    anon_df['customer_id'] = [str(uuid.uuid4()) for _ in range(len(anon_df))]
anon_df = anon_df.drop(columns=[c for c in ['residence','current_location','employer'] if c in anon_df.columns])
anon_df.info()

In [ ]:
# Save locally only; do not upload this customer-level output to a public repository
anon_df.to_csv('anonymised_customers.csv', index=False)